# Jalkapallon ennustemalli — koko putki vaihe vaiheelta

Tämä muistikirja käy läpi:
1. Dataa haetaan FBref:stä (pohjadata), Understat:sta (xG-trendit) ja StatsBombista (tapahtumadata)
2. Piirteet rakennetaan rolling-form -tyyliin
3. Dixon-Coles -malli sovitetaan ja siitä tuotetaan ennusteita
4. LightGBM-mallia verrataan Dixon-Colesiin
5. Pelaajatason ennusteet
6. SofaScore live-pollaus -esimerkki

> **Vinkki:** aja yksi solu kerrallaan ja lue kommentit. Ensimmäisellä ajolla data lataa hetken — soccerdata cachettää HTML-sivut, joten myöhemmät ajot ovat nopeita.

## 0. Asetukset

In [2]:
# Lisätään projektin juuri Python-polkuun, jotta voimme importata `src`-paketin
import sys
from pathlib import Path

# Notebookin kansio on .../football-prediction/notebooks → juuri yksi taso ylös
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 100

import config

## 1. Hae FBref-pohjadata

Aluksi vain Premier League viime kaudelta — kokeile sitten muita.

In [3]:
from src.data.fbref import lataa_otteludata, lataa_joukkueen_kausistatistiikka, vie_csv

LIIGAT = ['ENG-Premier League']
KAUDET = ['2324', '2425']

ottelut = lataa_otteludata(LIIGAT, KAUDET, cache_dir=config.RAW_DATA_DIR / 'fbref')
print(f'Haettiin {len(ottelut)} ottelua')
ottelut.head()

[04/26/26 13:29:43] INFO     No custom team name replacements found. You can configure these in       ]8;id=12333331;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=12333332;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_config.py#92\92]8;;\
                             C:\Users\vvsaa\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=12333338;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=12333339;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_config.py#190\190]8;;\
                             C:\Users\vvsaa\soccerdata\config\league_dict.json.                                    

                    INFO     Saving cached data to                                                   ]8;id=12333346;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12333347;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\
                             C:\Users\vvsaa\Documents\football-prediction\data\raw\fbref                           

Haettiin 760 ottelua


,league,season,game,week,day,date,time,home_team,score,away_team,attendance,venue,referee,match_report,notes,game_id
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,1,Fri,2023-08-11,20:00 (22:00),Burnley,0–3,Manchester City,21572,Turf Moor,Craig Pawson,/en/matches/3a6836b4/Burnley-Manchester-City-A...,<NA>,3a6836b4
1,ENG-Premier League,2324,2023-08-12 Arsenal-Nottingham Forest,1,Sat,2023-08-12,12:30 (14:30),Arsenal,2–1,Nottingham Forest,59984,Emirates Stadium,Michael Oliver,/en/matches/26a7f90c/Arsenal-Nottingham-Forest...,<NA>,26a7f90c
2,ENG-Premier League,2324,2023-08-12 Bournemouth-West Ham United,1,Sat,2023-08-12,15:00 (17:00),Bournemouth,1–1,West Ham United,11245,Vitality Stadium,Peter Bankes,/en/matches/d6bbf293/Bournemouth-West-Ham-Unit...,<NA>,d6bbf293
3,ENG-Premier League,2324,2023-08-12 Brighton-Luton Town,1,Sat,2023-08-12,15:00 (17:00),Brighton,4–1,Luton Town,31872,The American Express Community Stadium,David Coote,/en/matches/56a137f7/Brighton-and-Hove-Albion-...,<NA>,56a137f7
4,ENG-Premier League,2324,2023-08-12 Everton-Fulham,1,Sat,2023-08-12,15:00 (17:00),Everton,0–1,Fulham,39940,Goodison Park,Stuart Attwell,/en/matches/15addfc7/Everton-Fulham-August-12-...,<NA>,15addfc7


In [4]:
# Vie CSV:nä Power BI:tä varten
vie_csv(ottelut, config.PROCESSED_DATA_DIR / 'pl_ottelut.csv')
print('CSV tallennettu kansioon', config.PROCESSED_DATA_DIR)

CSV tallennettu kansioon C:\Users\vvsaa\Documents\football-prediction\data\processed


In [5]:
# Joukkuetason kausitilastot — xG, xGA jne.
joukkue_stats = lataa_joukkueen_kausistatistiikka(LIIGAT, ['2425'], stat_type='standard',
                                                  cache_dir=config.RAW_DATA_DIR / 'fbref')
joukkue_stats.head()

[04/26/26 13:29:52] INFO     Saving cached data to                                                   ]8;id=12333352;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12333353;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\
                             C:\Users\vvsaa\Documents\football-prediction\data\raw\fbref                           

league season         team players_used   Age  Poss Playing Time                  Performance                                  Per 90 Minutes                           \
                                                                             MP Starts   Min 90s         Gls Ast  G+A G-PK PK PKatt CrdY CrdR            Gls   Ast   G+A  G-PK G+A-PK   
0  ENG-Premier League   2425      Arsenal           25  25.8  56.9           38    418  3420  38          67  55  122   65  2     2   70    6           1.76  1.45  3.21  1.71   3.16   
1  ENG-Premier League   2425  Aston Villa           28  27.0  50.5           38    418  3420  38          56  45  101   53  3     6   76    4           1.47  1.18  2.66  1.39   2.58   
2  ENG-Premier League   2425  Bournemouth           29  25.1  48.5           38    418  3420  38          57  41   98   51  6     7   97    3            1.5  1.08  2.58  1.34   2.42   
3  ENG-Premier League   2425    Brentford           28  25.8  47.9           38    418  3420  38          65  44  109   60  5     6   62    1           1.71  1.16  2.87  1.58   2.74   
4  ENG-Premier League   2425     Brighton           32  24.8  52.3           38    418  3420  38          64  41  105   57  7     7   78    3           1.68  1.08  2.76   1.5   2.58   

                                                 url  
                                                      
0        /en/squads/18bb7c10/2024-2025/Arsenal-Stats  
1    /en/squads/8602292d/2024-2025/Aston-Villa-Stats  
2    /en/squads/4ba7cbea/2024-2025/Bournemouth-Stats  
3      /en/squads/cd051869/2024-2025/Brentford-Stats  
4  /en/squads/d07537b9/2024-2025/Brighton-and-Hov...

## 2. Hae Understat — xG-trendit

In [6]:
from src.data.understat import lataa_otteludata as lataa_us_ottelut
from src.data.understat import lataa_laukaukset, joukkueen_xg_aikasarja

us_ottelut = lataa_us_ottelut(['ENG-Premier League'], ['2425'],
                              cache_dir=config.RAW_DATA_DIR / 'understat')
us_ottelut.head()

[04/26/26 13:29:57] INFO     Saving cached data to                                                   ]8;id=12333358;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12333359;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\
                             C:\Users\vvsaa\Documents\football-prediction\data\raw\understat                       

[2026-04-26 13:29:57] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=12333366;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12333367;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\                 
                             tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                              

,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
0,ENG-Premier League,2425,2024-08-16 Manchester United-Fulham,1,2024,26602,2024-08-16 19:00:00,89,228,Manchester United,Fulham,FLH,MUN,1,0,2.04268,0.418711,True,True,https://understat.com/match/26602
1,ENG-Premier League,2425,2024-08-17 Arsenal-Wolverhampton Wanderers,1,2024,26604,2024-08-17 14:00:00,83,229,Arsenal,Wolverhampton Wanderers,WOL,ARS,2,0,1.6283,0.575835,True,True,https://understat.com/match/26604
2,ENG-Premier League,2425,2024-08-17 Everton-Brighton,1,2024,26605,2024-08-17 14:00:00,72,220,Everton,Brighton,BRI,EVE,0,3,0.405325,1.79083,True,True,https://understat.com/match/26605
3,ENG-Premier League,2425,2024-08-17 Ipswich-Liverpool,1,2024,26603,2024-08-17 11:30:00,285,87,Ipswich,Liverpool,LIV,IPS,0,2,0.342601,3.92906,True,True,https://understat.com/match/26603
4,ENG-Premier League,2425,2024-08-17 Newcastle United-Southampton,1,2024,26606,2024-08-17 14:00:00,86,74,Newcastle United,Southampton,SOU,NEW,1,0,0.433489,1.95483,True,True,https://understat.com/match/26606


In [7]:
# Laukaustason data — paljon dataa, voi kestää hetken
laukaukset = lataa_laukaukset(['ENG-Premier League'], ['2425'],
                              cache_dir=config.RAW_DATA_DIR / 'understat')
print(f'Haettiin {len(laukaukset):,} laukausta')
laukaukset.head()

[04/26/26 13:30:03] INFO     Saving cached data to                                                   ]8;id=12333372;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12333373;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\
                             C:\Users\vvsaa\Documents\football-prediction\data\raw\understat                       

Haettiin 9,878 laukausta


,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,team_id,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
0,ENG-Premier League,2425,2024-08-16 Manchester United-Fulham,Fulham,Adama Traoré,1,2024,26602,2024-08-16 19:00:00,584627,228,900,666195,Rodrigo Muniz,0.036473,0.867,0.278,7,Right Foot,Open Play,Missed Shot
1,ENG-Premier League,2425,2024-08-16 Manchester United-Fulham,Fulham,Adama Traoré,1,2024,26602,2024-08-16 19:00:00,584644,228,900,666191,Sasa Lukic,0.057856,0.809,0.512,70,Right Foot,Open Play,Blocked Shot
2,ENG-Premier League,2425,2024-08-16 Manchester United-Fulham,Fulham,Adama Traoré,1,2024,26602,2024-08-16 19:00:00,584645,228,900,666190,Andreas Pereira,0.020298,0.758,0.361,71,Right Foot,Open Play,Blocked Shot
3,ENG-Premier League,2425,2024-08-16 Manchester United-Fulham,Fulham,Alex Iwobi,1,2024,26602,2024-08-16 19:00:00,584636,228,500,666190,Andreas Pereira,0.018718,0.962,0.628,47,<NA>,From Corner,Missed Shot
4,ENG-Premier League,2425,2024-08-16 Manchester United-Fulham,Fulham,Calvin Bassey,1,2024,26602,2024-08-16 19:00:00,584646,228,11728,666195,Rodrigo Muniz,0.034208,0.823,0.572,75,Left Foot,From Corner,Blocked Shot


In [8]:
# Visualisoi yhden joukkueen xG-trendi
from src.viz.xg_plots import plot_rolling_xg

JOUKKUE = 'Arsenal'
aikasarja = joukkueen_xg_aikasarja(laukaukset, JOUKKUE)
fig, ax = plt.subplots(figsize=(10, 5))
plot_rolling_xg(aikasarja, JOUKKUE, ikkuna=5, ax=ax)
plt.tight_layout()
plt.show()

KeyError: "Label(s) ['xG'] do not exist"

## 3. StatsBomb-tapahtumadata

StatsBombilla ei ole top-5 -liigojen kaudessa olevaa avointa kattausta, mutta MM-kisat ja muut turnaukset ovat saatavilla. Käytetään niitä "ground truthina" mallin xG-arvoille.

In [9]:
from src.data.statsbomb import listaa_kilpailut, hae_ottelut, hae_tapahtumat, laske_xg_per_joukkue

kilpailut = listaa_kilpailut()
kilpailut.head(20)

[04/26/26 13:30:52] WARNING  C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packag ]8;id=12333380;file://C:\Users\vvsaa\AppData\Local\Python\pythoncore-3.14-64\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=12333381;file://C:\Users\vvsaa\AppData\Local\Python\pythoncore-3.14-64\Lib\_py_warnings.py#230\230]8;;\
                             es\statsbombpy\api_client.py:21: NoAuthWarning: credentials were                      
                             not supplied. open data access only                                                   
                               warnings.warn(                                                                      
                                                                                                                   

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,NaN,NaN,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,NaN,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882
5,16,2,Europe,Champions League,male,False,False,2016/2017,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,NaN,2024-02-13T02:37:32.205154
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,NaN,2024-06-12T07:45:38.786894
7,16,26,Europe,Champions League,male,False,False,2014/2015,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,NaN,2024-02-12T12:49:54.914228
8,16,25,Europe,Champions League,male,False,False,2013/2014,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,NaN,2024-02-12T12:48:48.479157
9,16,24,Europe,Champions League,male,False,False,2012/2013,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,NaN,2024-02-12T12:47:34.340413


In [10]:
# Esimerkki: MM 2022 finaali (Argentiina vs Ranska)
# competition_id = 43 (FIFA World Cup), season_id = 106 (2022)
mm_ottelut = hae_ottelut(competition_id=43, season_id=106)
finaali = mm_ottelut.sort_values('match_date').iloc[-1]
print('Finaali:', finaali['home_team'], 'vs', finaali['away_team'])

events = hae_tapahtumat(match_id=finaali['match_id'])
print(f'{len(events):,} tapahtumaa')
laske_xg_per_joukkue(events)

[04/26/26 13:30:54] WARNING  C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packag ]8;id=12333386;file://C:\Users\vvsaa\AppData\Local\Python\pythoncore-3.14-64\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=12333387;file://C:\Users\vvsaa\AppData\Local\Python\pythoncore-3.14-64\Lib\_py_warnings.py#230\230]8;;\
                             es\statsbombpy\api_client.py:21: NoAuthWarning: credentials were                      
                             not supplied. open data access only                                                   
                               warnings.warn(                                                                      
                                                                                                                   

Finaali: Argentina vs France
4,407 tapahtumaa


,team,shots,xG,goals
0,Argentina,24,5.892306,7
1,France,14,5.406618,5


## 4. Piirteiden rakentaminen

Tehdään rolling-form Premier Leaguen otteluista.

In [11]:
from src.features.team_features import (
    laajenna_per_joukkue, rolling_keskiarvo, yhdista_ottelutasolle,
    lisaa_1x2, lisaa_total_goals
)

# Yhdistetään Understat-data joka sisältää xG:n
us_ottelut_clean = us_ottelut.dropna(subset=['home_score', 'away_score']).copy()
us_ottelut_clean['date'] = pd.to_datetime(us_ottelut_clean['date'])

# Per-joukkue muotoon (2 riviä per ottelu)
joukkue_ottelu = laajenna_per_joukkue(us_ottelut_clean)

# xG-piirteet (Understat tarjoaa home_xg ja away_xg, normalisoidaan goals_for tyyliin)
joukkue_ottelu['xg_for'] = np.where(
    joukkue_ottelu['is_home'] == 1, joukkue_ottelu['home_xg'], joukkue_ottelu['away_xg'])
joukkue_ottelu['xg_against'] = np.where(
    joukkue_ottelu['is_home'] == 1, joukkue_ottelu['away_xg'], joukkue_ottelu['home_xg'])

# Rolling-piirteet
piirteet = ['goals_for', 'goals_against', 'xg_for', 'xg_against']
joukkue_ottelu = rolling_keskiarvo(joukkue_ottelu, piirteet, ikkuna=5)

# Takaisin ottelutasolle
rolling_sarakkeet = [f'{p}_rolling5' for p in piirteet]
ottelutaso = yhdista_ottelutasolle(joukkue_ottelu, rolling_sarakkeet)
ottelutaso = lisaa_1x2(ottelutaso)
ottelutaso = lisaa_total_goals(ottelutaso)
ottelutaso.head()

KeyError: ['home_score', 'away_score']

## 5. Dixon-Coles -malli

Sovitetaan Poisson-malli aikapainolla — uudet ottelut painavat enemmän.

In [12]:
from src.models.dixon_coles import DixonColesModel

# Käytetään kaikkia ratkenneita otteluita
treenidata = us_ottelut_clean.dropna(subset=['home_score', 'away_score']).copy()

malli = DixonColesModel().fit(
    treenidata,
    home_team_col='home_team',
    away_team_col='away_team',
    home_goals_col='home_score',
    away_goals_col='away_score',
    decay=0.0065,
    date_col='date',
)
print(f'Kotietu (gamma): {malli.home_advantage:.3f}')
print(f'Rho-korjaus: {malli.rho:.3f}')
print(f'\\nVahvimmat hyökkäysreitingit:')
for joukkue, attack in sorted(malli.attack.items(), key=lambda x: -x[1])[:5]:
    print(f'  {joukkue}: {attack:.3f}')

NameError: name 'us_ottelut_clean' is not defined

In [13]:
# Esimerkkiennuste
KOTI = 'Manchester City'
VIERAS = 'Arsenal'

lam, mu = malli.expected_goals(KOTI, VIERAS)
print(f'Odotetut maalit: {KOTI} {lam:.2f} — {mu:.2f} {VIERAS}\n')

print('1X2-todennäköisyydet:')
for k, v in malli.predict_1x2(KOTI, VIERAS).items():
    print(f'  {k:6s} {v:.3f}')

print('\nOver/Under 2.5:')
for k, v in malli.predict_over_under(KOTI, VIERAS, line=2.5).items():
    print(f'  {k:6s} {v:.3f}')

print('\nBTTS:')
for k, v in malli.predict_btts(KOTI, VIERAS).items():
    print(f'  {k:10s} {v:.3f}')

print('\nTodennäköisimmät tarkat tulokset:')
for tulos, p in malli.todennakoisin_tulos(KOTI, VIERAS, top_n=5):
    print(f'  {tulos}: {p:.3f}')

NameError: name 'malli' is not defined

## 6. LightGBM-malli rolling-piirteistä

Vertaillaan Dixon-Colesia gradient boosting -malliin.

In [14]:
from src.models.outcome_model import opeta_1x2, opeta_over_under
from sklearn.metrics import log_loss, accuracy_score

# Poistetaan ne rivit joissa rolling-piirteet eivät ole vielä valmiita (kauden alku)
feature_cols = [c for c in ottelutaso.columns if 'rolling5' in c]
data = ottelutaso.dropna(subset=feature_cols).copy().sort_values('date')

# Aikajaettu split: ensimmäinen 80% treenaukseen, viimeinen 20% validointiin
split = int(len(data) * 0.8)
train, valid = data.iloc[:split], data.iloc[split:]

X_train, y_train = train[feature_cols], train['result_1x2']
X_valid, y_valid = valid[feature_cols], valid['result_1x2']

lgb_1x2 = opeta_1x2(X_train, y_train, X_valid, y_valid)
ennusteet = lgb_1x2.predict(X_valid)
print(f'Validointi log loss: {log_loss(y_valid, ennusteet):.3f}')
print(f'Validointi accuracy: {accuracy_score(y_valid, ennusteet.argmax(axis=1)):.3f}')

NameError: name 'ottelutaso' is not defined

## 7. Pelaajatason ennusteet

Yksinkertainen baseline.

In [15]:
from src.data.fbref import lataa_pelaajat_kausi
from src.features.player_features import per_90
from src.models.player_model import ennusta_per90_baseline, todennakoisyys_pelaaja_skoraa

pelaajat = lataa_pelaajat_kausi(['ENG-Premier League'], ['2425'],
                                stat_type='standard',
                                cache_dir=config.RAW_DATA_DIR / 'fbref')
pelaajat.head()

[04/26/26 13:31:24] INFO     Saving cached data to                                                   ]8;id=12333392;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12333393;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\
                             C:\Users\vvsaa\Documents\football-prediction\data\raw\fbref                           

[04/26/26 13:31:55] ERROR    Error while scraping                                                    ]8;id=12333399;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12333400;file://C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\soccerdata\_common.py#623\623]8;;\
                             https://fbref.com/en/comps/9/2024-2025/stats/2024-2025-Premier-League-S               
                             tats. Retrying in 0 seconds... (attempt 1 of 5).                                      
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\s               
                             occerdata\_common.py", line 607, in _download_and_save                                
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ~~~~~~~~~~~~~~~~~~~^^^^^                                               
                               File                                                                                
                             "C:\Users\vvsaa\Documents\football-prediction\.venv\Lib\site-packages\s               
                             occerdata\fbref.py", line 1031, in _validate_page                                     
                                 raise Exception(                                                                  
                                 ...<2 lines>...                                                                   
                                 )                                                                                 
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

league season     team         player nation    pos age  born Playing Time                    Performance                                 Per 90 Minutes                         
                                                                                       MP Starts   Min   90s         Gls Ast G+A G-PK PK PKatt CrdY CrdR            Gls   Ast   G+A  G-PK G+A-PK
0  ENG-Premier League   2425  Arsenal      Ben White    ENG     DF  26  1997           17     13  1198  13.3           0   2   2    0  0     0    2    0            0.0  0.15  0.15   0.0   0.15
1  ENG-Premier League   2425  Arsenal    Bukayo Saka    ENG  FW,MF  22  2001           25     20  1729  19.2           6  10  16    5  1     1    3    0           0.31  0.52  0.83  0.26   0.78
2  ENG-Premier League   2425  Arsenal     David Raya    ESP     GK  28  1995           38     38  3420  38.0           0   0   0    0  0     0    3    0            0.0   0.0   0.0   0.0    0.0
3  ENG-Premier League   2425  Arsenal    Declan Rice    ENG     MF  25  1999           35     33  2825  31.4           4   7  11    4  0     0    7    1           0.13  0.22  0.35  0.13   0.35
4  ENG-Premier League   2425  Arsenal  Ethan Nwaneri    ENG  FW,MF  17  2007           26     11   895   9.9           4   2   6    4  0     0    1    0            0.4   0.2   0.6   0.4    0.6

In [16]:
# soccerdata palauttaa MultiIndex-sarakkeet — katsotaan rakenne
# Ja löydetään relevantit sarakkeet (xG, xA, minuutit)
print('Saatavilla olevat sarakkeet:')
print(list(pelaajat.columns)[:30])

Saatavilla olevat sarakkeet:
[('league', ''), ('season', ''), ('team', ''), ('player', ''), ('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('Playing Time', 'MP'), ('Playing Time', 'Starts'), ('Playing Time', 'Min'), ('Playing Time', '90s'), ('Performance', 'Gls'), ('Performance', 'Ast'), ('Performance', 'G+A'), ('Performance', 'G-PK'), ('Performance', 'PK'), ('Performance', 'PKatt'), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Per 90 Minutes', 'Gls'), ('Per 90 Minutes', 'Ast'), ('Per 90 Minutes', 'G+A'), ('Per 90 Minutes', 'G-PK'), ('Per 90 Minutes', 'G+A-PK')]


Sarakkeiden nimet vaihtelevat soccerdatan version mukaan — tarkista `pelaajat.columns` ja säädä alla olevia sarakenimet vastaaviksi.

Esim. tyypilliset nimet ovat `Performance Gls`, `Expected xG`, `Expected xAG`, `Playing Time Min`.

In [17]:
# Esimerkki — sovita sarakenimet oman datasi mukaan
p = pelaajat.copy()
# Käytetään turvallista syntaksia, joka toimii sekä MultiIndex- että flat-sarakkeissa
p.columns = ['_'.join([str(x) for x in c]).strip('_') if isinstance(c, tuple) else c
             for c in p.columns]
print('Sarakkeet flat-muodossa:')
for c in p.columns:
    if 'xG' in c or 'Min' in c or 'Gls' in c:
        print(' ', c)

Sarakkeet flat-muodossa:
  Playing Time_Min
  Performance_Gls
  Per 90 Minutes_Gls
  Per 90 Minutes_Ast
  Per 90 Minutes_G+A
  Per 90 Minutes_G-PK
  Per 90 Minutes_G+A-PK


In [18]:
# Kun sarakenimet ovat selvillä — esim:
# minuutit_col = 'Playing Time_Min'
# xg_col = 'Expected_xG'
# xa_col = 'Expected_xAG'
# Säädä omasi vastaavasti.

# Demo: jos sarakkeita ei löydy, käytetään numpy-satunnaisarvoja näytteen luomiseksi
import numpy as np
rng = np.random.default_rng(42)
demo = p.head(20).copy()
demo['minutes'] = rng.integers(500, 2700, len(demo))
demo['xG'] = rng.uniform(0, 12, len(demo))
demo['xA'] = rng.uniform(0, 8, len(demo))

demo90 = per_90(demo, ['xG', 'xA'], minuutit_col='minutes')
demo90[['xG_per90', 'xA_per90']].head()

,xG_per90,xA_per90
0,0.575376,0.770444
1,0.454544,0.316352
2,0.358440,0.120925
3,0.606541,0.182069
4,0.329812,0.232838


In [19]:
# Ennusta seuraavan ottelun arvo (oletetaan 75 min peliaika, neutraali vastustaja)
ennusteet = ennusta_per90_baseline(
    demo90,
    odotetut_minuutit=pd.Series(75, index=demo90.index),
    metriikat=('xG_per90', 'xA_per90'),
)
ennusteet[['expected_xG', 'expected_xA']].head()

,expected_xG,expected_xA
0,0.479480,0.642036
1,0.378787,0.263627
2,0.298700,0.100771
3,0.505451,0.151724
4,0.274844,0.194031


In [20]:
# 'Anytime scorer' -todennäköisyys
for idx, rivi in ennusteet.head().iterrows():
    p = todennakoisyys_pelaaja_skoraa(rivi['expected_xG'])
    print(f"  Pelaaja idx={idx}: anytime scorer = {p['anytime_scorer']:.3f}")

  Pelaaja idx=0: anytime scorer = 0.381
  Pelaaja idx=1: anytime scorer = 0.315
  Pelaaja idx=2: anytime scorer = 0.258
  Pelaaja idx=3: anytime scorer = 0.397
  Pelaaja idx=4: anytime scorer = 0.240


## 8. Live-seuranta SofaScoresta

Pollataan käynnissä olevia otteluita.

> ⚠️ Älä aja tätä silmukkaa loputtomasti — käytä kohtuudella.

In [21]:
from src.data.sofascore import hae_live_ottelut, parsi_live_ottelut

events = hae_live_ottelut()
live = parsi_live_ottelut(events)
print(f'Käynnissä {len(live)} ottelua')
live.head(10)

HTTPError: 403 Client Error: Forbidden for url: https://api.sofascore.com/api/v1/sport/football/events/live

## 9. Mihin tästä eteenpäin?

- Lisää piirteitä: lepopäivät, pelipaikkamuutokset, sääolot
- Walk-forward CV (`sklearn.model_selection.TimeSeriesSplit`)
- Kalibroi todennäköisyydet `sklearn.calibration.CalibratedClassifierCV`-luokalla
- Vertaa markkinakertoimiin → Kelly-criterion -panostuskoko
- Vie data Power BI:hin: kansiossa `data/processed/` on CSV-tiedostot, joista voit Power BI:ssä rakentaa dashboardin